# 02 — Embedding Space Analysis

Compare raw BGE-M3 embeddings vs GNN-enriched embeddings.
Visualize clustering, inter/intra-note similarity, and GNN impact.

**Requires:** `vault-exec init` (KV with embeddings)

In [1]:
import { openVaultStore } from "../src/db/index.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const db = await openVaultStore(DB_PATH);
const allNotes = await db.getAllNotes();

console.log(`Loaded ${allNotes.length} notes from KV`);
console.log(`With raw embedding: ${allNotes.filter(n => n.embedding).length}`);
console.log(`With GNN embedding: ${allNotes.filter(n => n.gnnEmbedding).length}`);


Loaded 124 notes from KV


With raw embedding: 124


With GNN embedding: 124


In [2]:
// Cosine similarity helper
function cosine(a: number[], b: number[]): number {
  let dot = 0, na = 0, nb = 0;
  for (let i = 0; i < a.length; i++) {
    dot += a[i] * b[i];
    na += a[i] * a[i];
    nb += b[i] * b[i];
  }
  return dot / (Math.sqrt(na) * Math.sqrt(nb) + 1e-9);
}

// L2 norm
function l2norm(v: number[]): number {
  return Math.sqrt(v.reduce((s, x) => s + x * x, 0));
}

// Pairwise cosine similarity matrix
const names = allNotes.map(n => n.name);
const rawEmbs = allNotes.map(n => n.embedding!);
const gnnEmbs = allNotes.map(n => n.gnnEmbedding ?? n.embedding!);

console.log(`Embedding dim: ${rawEmbs[0]?.length}`);

Embedding dim: 1024


In [3]:
// Raw vs GNN embedding comparison per note
const comparison = allNotes.map((n, i) => {
  const raw = n.embedding;
  const gnn = n.gnnEmbedding;
  if (!raw || !gnn) return null;
  return {
    name: n.name,
    level: n.level,
    rawNorm: l2norm(raw).toFixed(4),
    gnnNorm: l2norm(gnn).toFixed(4),
    cosSim: cosine(raw, gnn).toFixed(4),
    l2Delta: Math.sqrt(raw.reduce((s, v, j) => s + (v - gnn[j]) ** 2, 0)).toFixed(4),
  };
}).filter(Boolean);

console.table(comparison);

const avgCos = comparison.reduce((s, c) => s + parseFloat(c!.cosSim), 0) / comparison.length;
const avgDelta = comparison.reduce((s, c) => s + parseFloat(c!.l2Delta), 0) / comparison.length;
console.log(`\nAvg cos(raw, GNN): ${avgCos.toFixed(4)}`);
console.log(`Avg L2 delta:      ${avgDelta.toFixed(4)}`);

┌───────┬──────────────────────────────────────┬───────┬──────────┬──────────┬──────────┬──────────┐
│ (idx) │ name                                 │ level │ rawNorm  │ gnnNorm  │ cosSim   │ l2Delta  │
├───────┼──────────────────────────────────────┼───────┼──────────┼──────────┼──────────┼──────────┤
│     0 │ "Account Contract Links"             │     1 │ "1.0000" │ "1.0000" │ "1.0000" │ "0.0000" │
│     1 │ "Account Health Inputs"              │     1 │ "1.0000" │ "1.0000" │ "1.0000" │ "0.0000" │
│     2 │ "Account Ticket Links"               │     1 │ "1.0000" │ "1.0000" │ "1.0000" │ "0.0000" │
│     3 │ "Account Tier Weight"                │     1 │ "1.0000" │ "1.0000" │ "1.0000" │ "0.0000" │
│     4 │ "Accounts"                           │     0 │ "1.0000" │ "1.0000" │ "1.0000" │ "0.0000" │
│     5 │ "Action Dispatch Queue"              │     9 │ "1.0000" │ "1.0000" │ "1.0000" │ "0.0000" │
│     6 │ "Action Eligibility"                 │     5 │ "1.0000" │ "1.0000" │ "1.0000" │ "


Avg cos(raw, GNN): 0.8273


Avg L2 delta:      0.3765


In [4]:
// Pairwise similarity heatmap (GNN-enriched embeddings)
// Rebuild aligned arrays to avoid stale/reassigned globals across reruns.
const heatmapPairs = allNotes
  .filter((n) => Array.isArray(n.gnnEmbedding ?? n.embedding))
  .map((n) => ({
    name: n.name,
    emb: (n.gnnEmbedding ?? n.embedding!) as number[],
  }));

const heatmapData: Array<{noteA: string, noteB: string, similarity: number}> = [];
for (let i = 0; i < heatmapPairs.length; i++) {
  for (let j = 0; j < heatmapPairs.length; j++) {
    heatmapData.push({
      noteA: heatmapPairs[i].name,
      noteB: heatmapPairs[j].name,
      similarity: parseFloat(cosine(heatmapPairs[i].emb, heatmapPairs[j].emb).toFixed(4)),
    });
  }
}

const heatmapSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Pairwise Cosine Similarity (GNN-enriched)",
  "width": 450,
  "height": 450,
  "data": { "values": heatmapData },
  "mark": "rect",
  "encoding": {
    "x": { "field": "noteA", "type": "nominal", "title": null },
    "y": { "field": "noteB", "type": "nominal", "title": null },
    "color": {
      "field": "similarity", "type": "quantitative",
      "scale": { "scheme": "viridis", "domain": [0.5, 1.0] },
      "title": "Cosine Sim"
    },
    "tooltip": [{ "field": "noteA" }, { "field": "noteB" }, { "field": "similarity", "format": ".3f" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": heatmapSpec }, { raw: true });

In [5]:
// GNN movement: how much did each note's embedding change?
const movementData = comparison!.map(c => ({
  name: c!.name,
  l2Delta: parseFloat(c!.l2Delta),
  cosSim: parseFloat(c!.cosSim),
  level: allNotes.find(n => n.name === c!.name)!.level,
}));

const movementSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "GNN Embedding Movement (Raw → Enriched)",
  "width": 500,
  "height": 300,
  "data": { "values": movementData },
  "mark": { "type": "bar" },
  "encoding": {
    "x": { "field": "name", "type": "nominal", "sort": "-y", "title": "Note" },
    "y": { "field": "l2Delta", "type": "quantitative", "title": "L2 Distance (raw → GNN)" },
    "color": { "field": "level", "type": "ordinal", "title": "DAG Level" },
    "tooltip": [{ "field": "name" }, { "field": "l2Delta", "format": ".4f" }, { "field": "cosSim", "format": ".4f" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": movementSpec }, { raw: true });

In [6]:
// Parent-child similarity: are dependent notes closer after GNN?
import { DenoVaultReader } from "../src/core/io.ts";
import { parseVault } from "../src/core/parser.ts";

const reader2 = new DenoVaultReader();
const notes2 = await parseVault(reader2, VAULT_PATH);
const embByName = new Map(allNotes.map((n) => [n.name, {
  raw: n.embedding as number[] | undefined,
  gnn: (n.gnnEmbedding ?? n.embedding) as number[] | undefined,
}]));

const pcPairs: Array<{parent: string, child: string, rawSim: number, gnnSim: number, delta: number}> = [];
for (const note of notes2) {
  for (const child of note.wikilinks) {
    const p = embByName.get(note.name);
    const c = embByName.get(child);
    if (p?.raw && p.gnn && c?.raw && c.gnn) {
      const rawSim = cosine(p.raw, c.raw);
      const gnnSim = cosine(p.gnn, c.gnn);
      pcPairs.push({ parent: note.name, child, rawSim, gnnSim, delta: gnnSim - rawSim });
    }
  }
}

console.log('\n=== Parent-Child Similarity ===');
console.table(pcPairs.map(p => ({
  ...p,
  rawSim: p.rawSim.toFixed(4),
  gnnSim: p.gnnSim.toFixed(4),
  delta: (p.delta >= 0 ? '+' : '') + p.delta.toFixed(4),
})));

const avgRaw = pcPairs.reduce((s, p) => s + p.rawSim, 0) / pcPairs.length;
const avgGnn = pcPairs.reduce((s, p) => s + p.gnnSim, 0) / pcPairs.length;
console.log(`\nAvg parent-child sim RAW: ${avgRaw.toFixed(4)}`);
console.log(`Avg parent-child sim GNN: ${avgGnn.toFixed(4)}`);
console.log(`Delta: ${(avgGnn - avgRaw >= 0 ? '+' : '')}${(avgGnn - avgRaw).toFixed(4)}`);


=== Parent-Child Similarity ===


┌───────┬──────────────────────────────────────┬──────────────────────────────────────┬──────────┬──────────┬───────────┐
│ (idx) │ parent                               │ child                                │ rawSim   │ gnnSim   │ delta     │
├───────┼──────────────────────────────────────┼──────────────────────────────────────┼──────────┼──────────┼───────────┤
│     0 │ "CRM - Follow-up Engine"             │ "CRM - Deals"                        │ "0.6641" │ "0.3151" │ "-0.3490" │
│     1 │ "CRM - Inbox-to-CRM"                 │ "SHARED - Segment Owner Routing"     │ "0.7136" │ "0.3953" │ "-0.3183" │
│     2 │ "CRM - Inbox-to-CRM"                 │ "CRM - Contacts"                     │ "0.6974" │ "0.3797" │ "-0.3176" │
│     3 │ "CRM - Pipeline"                     │ "CRM - Deals"                        │ "0.7125" │ "0.3477" │ "-0.3648" │
│     4 │ "CRM - Daily Command Board"          │ "CRM - Pipeline"                     │ "0.7232" │ "0.6265" │ "-0.0967" │
│     5 │ "CRM - Daily C


Avg parent-child sim RAW: 0.6822


Avg parent-child sim GNN: 0.5835


Delta: -0.0987


In [7]:
db.close();
console.log('Done');

Done
